# 3. Boiling point: frozen molecular foundation-model transfer

    **Demo purpose:** contrast a task-specific GNN with a pretrained representation. MoLFormer is a molecular language model trained on a large collection of SMILES. It does not natively output boiling point, so this notebook uses its **frozen embeddings** and transfers labels from nearby training molecules.

    This is zero-update transfer—not strict label-free numerical zero-shot prediction—because the nearest-neighbor step still uses known training boiling points.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_SIZE = 1000
MODEL_ID = 'ibm-research/MoLFormer-XL-both-10pct'
BATCH_SIZE = 32
NEIGHBORS = 7
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

acquired_file = Path('outputs/nist_acquisition/NIST_BP_SMILES.parquet')
data_file = acquired_file if acquired_file.exists() else Path('data/CompoundData.parquet')
data = pd.read_parquet(data_file)
print(f'Using data source: {data_file}')
data = data.dropna(subset=['SMILES', 'BP']).drop_duplicates('SMILES')[['SMILES', 'BP']]
data = data.sample(n=min(SAMPLE_SIZE, len(data)), random_state=SEED).reset_index(drop=True)
train_data, test_data = train_test_split(data, test_size=0.20, random_state=SEED)
print(f'Using {len(train_data):,} training and {len(test_data):,} test molecules on {DEVICE}.')


In [ ]:
try:
    from transformers import AutoModel, AutoTokenizer
except ImportError as error:
    raise ImportError('Install the workshop environment; it includes transformers.') from error

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, deterministic_eval=True, trust_remote_code=True).to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad = False

@torch.inference_mode()
def embed(smiles):
    vectors = []
    for start in range(0, len(smiles), BATCH_SIZE):
        tokens = tokenizer(smiles[start:start + BATCH_SIZE], padding=True, truncation=True,
                           max_length=202, return_tensors='pt').to(DEVICE)
        outputs = model(**tokens)
        pooled = getattr(outputs, 'pooler_output', None)
        if pooled is None:
            hidden = outputs.last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
        vectors.append(pooled.float().cpu().numpy())
    return np.vstack(vectors)

train_embedding = embed(train_data.SMILES.tolist())
test_embedding = embed(test_data.SMILES.tolist())
print('Embedding shapes:', train_embedding.shape, test_embedding.shape)


In [ ]:
def normalize_rows(values):
    return values / np.linalg.norm(values, axis=1, keepdims=True).clip(1e-12)

reference = normalize_rows(train_embedding)
queries = normalize_rows(test_embedding)
similarity = queries @ reference.T
k = min(NEIGHBORS, len(train_data))
nearest = np.argpartition(-similarity, kth=k - 1, axis=1)[:, :k]
nearest_similarity = np.take_along_axis(similarity, nearest, axis=1)
weights = 1 / np.clip(1 - nearest_similarity, 1e-6, None)
targets = train_data.BP.to_numpy()[nearest]
predicted = (weights * targets).sum(axis=1) / weights.sum(axis=1)
actual = test_data.BP.to_numpy()

summary = pd.DataFrame([{
    'method': f'frozen MoLFormer + {k}-NN transfer',
    'MAE (K)': mean_absolute_error(actual, predicted),
    'RMSE (K)': mean_squared_error(actual, predicted) ** 0.5,
    'R²': r2_score(actual, predicted),
}])
display(summary.round(2))


## Interpretation

    This small comparison illustrates two current state-of-the-art approaches rather than a permanent ranking. The GNN learns directly from the boiling-point task; the foundation model brings a broad pretrained chemical representation. In either case, compare models on the same split before making a performance claim.
